In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [0]:
from pyspark.sql.functions import col

In [0]:
df_laptimes = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/lap_times.csv",
    header=True
)

df = df_laptimes.select(

    col("raceId").cast("int"),
    col("driverId").cast("int"),
    col("lap").cast("int"),
    col("milliseconds").cast("double")
).dropna()

pdf = df.toPandas()

X = pdf[["raceId", "driverId", "lap"]]
y = pdf["milliseconds"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

1. Create two (2) new tables in your own databse where you'll store the predictions from each model for this exercise.

In [0]:
spark.sql("USE workspace.default")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS random_forest_predictions (
    raceId INT,
    driverId INT,
    lap INT,
    actual_milliseconds DOUBLE,
    predicted_milliseconds DOUBLE
)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS gradient_boosting_predictions (
    raceId INT,
    driverId INT,
    lap INT,
    actual_milliseconds DOUBLE,
    predicted_milliseconds DOUBLE
)
""")

2. Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments

In [0]:
def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    return preds, mae, mse, rmse, r2

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [0]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    random_state=42
)

with mlflow.start_run(run_name="Random Forest Model"):
    rf_model.fit(X_train, y_train)

    rf_preds, rf_mae, rf_mse, rf_rmse, rf_r2 = evaluate_model(
        rf_model, X_test, y_test
    )

    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 12)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("mae", rf_mae)
    mlflow.log_metric("mse", rf_mse)
    mlflow.log_metric("rmse", rf_rmse)
    mlflow.log_metric("r2", rf_r2)

    plt.figure()
    plt.scatter(y_test, rf_preds)
    plt.xlabel("Actual Lap Time")
    plt.ylabel("Predicted Lap Time")
    plt.title("Random Forest Actual vs Predicted")
    plt.savefig("rf_prediction_plot.png")
    mlflow.log_artifact("rf_prediction_plot.png")

    rf_importance = pd.DataFrame({
        "feature": X.columns,
        "importance": rf_model.feature_importances_
    })
    rf_importance.to_csv("rf_feature_importance.csv", index=False)
    mlflow.log_artifact("rf_feature_importance.csv")

    mlflow.sklearn.log_model(rf_model, "random_forest_model")

In [0]:
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

with mlflow.start_run(run_name="Gradient Boosting Model"):
    gb_model.fit(X_train, y_train)

    gb_preds, gb_mae, gb_mse, gb_rmse, gb_r2 = evaluate_model(
        gb_model, X_test, y_test
    )

    mlflow.log_param("model_type", "GradientBoostingRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 3)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("mae", gb_mae)
    mlflow.log_metric("mse", gb_mse)
    mlflow.log_metric("rmse", gb_rmse)
    mlflow.log_metric("r2", gb_r2)

    plt.figure()
    plt.scatter(y_test, gb_preds)
    plt.xlabel("Actual Lap Time")
    plt.ylabel("Predicted Lap Time")
    plt.title("Gradient Boosting Actual vs Predicted")
    plt.savefig("gb_prediction_plot.png")
    mlflow.log_artifact("gb_prediction_plot.png")

    gb_importance = pd.DataFrame({
        "feature": X.columns,
        "importance": gb_model.feature_importances_
    })
    gb_importance.to_csv("gb_feature_importance.csv", index=False)
    mlflow.log_artifact("gb_feature_importance.csv")

    mlflow.sklearn.log_model(gb_model, "gradient_boosting_model")

3. For each model, store its predictions in the corresponding table you created in your own database. Ensure you are using your own database to store your predictions.

In [0]:
# Random Forest predictions output
rf_output = X_test.copy()
rf_output["actual_milliseconds"] = y_test.values
rf_output["predicted_milliseconds"] = rf_preds

# Gradient Boosting predictions output
gb_output = X_test.copy()
gb_output["actual_milliseconds"] = y_test.values
gb_output["predicted_milliseconds"] = gb_preds

In [0]:
# convert pandas DataFrames to Spark DataFrames
rf_spark = spark.createDataFrame(rf_output)
gb_spark = spark.createDataFrame(gb_output)

In [0]:
# store predictions in default database tables
rf_spark.write.mode("overwrite").saveAsTable(
    "default.random_forest_predictions"
)

gb_spark.write.mode("overwrite").saveAsTable(
    "default.gradient_boosting_predictions"
)

In [0]:
# check saved predictions
display(spark.table("default.random_forest_predictions"))
display(spark.table("default.gradient_boosting_predictions"))